# Cadence Financial Group — Enterprise PII Guard
## Case Study 2: Zero-Leakage Text Sanitization & Verification Architecture

This interactive walkthrough demonstrates the end-to-end execution of the **PII Guard** pipeline across Cadence's multi-format artifacts:
1. **`Cadence_Financial_Group_Organizational_Pack.pptx`** (9-slide executive briefing with employee emails and phone direct-dials)
2. **`Cadence_TPRM_Training.docx`** (402-paragraph clean third-party risk management manual — negative control)
3. **`Cadence_Group_Risk_Management_Policy__v6.0.pdf`** (35-page scanned risk policy document processed via adaptive parallel OCR)

### Core Security Invariants
- **Zero LLM Dependency**: 100% deterministic regex rules and algorithmic validators (Luhn, SSA, IPv4, ISO phone).
- **Fail-Closed Release Gate**: A document is released strictly if `verification_status == 'PASS'` and `release_allowed == True`.
- **Structure Preservation**: $\ge 80\%$ document layout and structure retention guaranteed post-redaction.

In [ ]:
import sys
from pathlib import Path

# Ensure source modules are on path
sys.path.insert(0, str(Path.cwd() / "src"))

from pii_guard.orchestration.pipeline import process_document
from pii_guard.reporting.audit import get_global_audit_trail

print("PII Guard pipeline components loaded successfully.")

## 1. Processing `Cadence_Financial_Group_Organizational_Pack.pptx`
Demonstrating multi-slide layout extraction, table cell parsing, and redaction of corporate emails and direct-dial phone extensions.

In [ ]:
pptx_result = process_document("Cadence_Financial_Group_Organizational_Pack.pptx")

print(f"Document ID:          {pptx_result.document_id}")
print(f"Workflow Status:      {pptx_result.workflow_status}")
print(f"Final Stage:          {pptx_result.current_stage}")
print(f"Verification Status:  {pptx_result.verification_status}")
print(f"Release Allowed:      {pptx_result.release_allowed}")
print(f"Redactions Applied:   {pptx_result.redactions_applied}")
print(f"Categories Detected:  {pptx_result.category_counts}")
print(f"Artifact Reference:   {pptx_result.artifact_ref}")

## 2. Processing `Cadence_TPRM_Training.docx` (Negative Control)
Demonstrating clean document handling with zero false positives across 402 paragraphs.

In [ ]:
docx_result = process_document("Cadence_TPRM_Training.docx")

print(f"Document ID:          {docx_result.document_id}")
print(f"Workflow Status:      {docx_result.workflow_status}")
print(f"Final Stage:          {docx_result.current_stage}")
print(f"Verification Status:  {docx_result.verification_status}")
print(f"Release Allowed:      {docx_result.release_allowed}")
print(f"Redactions Applied:   {docx_result.redactions_applied} (0 expected for clean document)")
print(f"Categories Detected:  {docx_result.category_counts}")

## 3. Processing `Cadence_Group_Risk_Management_Policy__v6.0.pdf`
Demonstrating 35-page scanned PDF extraction via multi-worker parallel OCR, large-scale entity classification (468 entities), and zero-leakage verification.

In [ ]:
pdf_result = process_document("Cadence_Group_Risk_Management_Policy__v6.0.pdf")

print(f"Document ID:          {pdf_result.document_id}")
print(f"Workflow Status:      {pdf_result.workflow_status}")
print(f"Final Stage:          {pdf_result.current_stage}")
print(f"Verification Status:  {pdf_result.verification_status}")
print(f"Release Allowed:      {pdf_result.release_allowed}")
print(f"Redactions Applied:   {pdf_result.redactions_applied}")
print(f"Categories Detected:  {pdf_result.category_counts}")
print(f"Artifact Reference:   {pdf_result.artifact_ref}")

## 4. Fail-Closed Security Boundary Demonstration
Demonstrating that any simulated residual leakage or verification anomaly immediately blocks document release and withholds artifact access tokens.

In [ ]:
blocked_result = process_document(
    "Cadence_Financial_Group_Organizational_Pack.pptx",
    options={"simulate_verification_failure": True}
)

print(f"Simulated Failure Workflow Status: {blocked_result.workflow_status}")
print(f"Current Stage:                    {blocked_result.current_stage}")
print(f"Verification Status:              {blocked_result.verification_status}")
print(f"Release Allowed:                  {blocked_result.release_allowed}")
print(f"Artifact Reference Withheld:       {blocked_result.artifact_ref is None}")
print(f"Sanitized Safe Error:             {blocked_result.safe_error}")

## 5. Tamper-Evident Audit Ledger Inspection
Inspecting the cryptographic event history recorded during document processing.

In [ ]:
trail = get_global_audit_trail()
events = trail.get_events_for_document(pdf_result.document_id)

print(f"Audit Events for PDF ({len(events)} events):")
for ev in events:
    print(f"  [{ev.timestamp.strftime('%H:%M:%S')}] {ev.stage.value:<10} | {ev.event_type:<24} | status={ev.status}")

# Export to Contract D compliance format
export_data = trail.export_audit_trail(pdf_result.document_id, format="json")
print(f"\nAudit export generated ({len(export_data)} bytes), ready for OneTrust integration.")